# BB84: detecting an eavesdropper with quantum physics

Post-quantum cryptography (what pqc-scan recommends) is ordinary software.
**Quantum key distribution** is a different, hardware-based idea: send key
bits as single photons. BB84 (Bennett & Brassard, 1984) is the classic
protocol. This simulation shows its key property: *measuring a quantum state
disturbs it*, so an eavesdropper leaves traces.

## The rules

* Alice encodes each bit in one of two **bases**: `+` (0 → horizontal,
  1 → vertical) or `x` (0 → 45°, 1 → 135°).
* Bob measures each photon in a random basis. Same basis → he reads Alice's
  bit. Different basis → he gets a random result.
* Afterwards they publish only the bases (not the bits) and keep the
  positions where the bases matched: the **sifted key**.

In [ ]:
import random

def measure(bit: int, sent_basis: str, measured_basis: str, rng: random.Random) -> int:
    return bit if sent_basis == measured_basis else rng.randint(0, 1)

def bb84(n: int, eve: bool, seed: int = 7) -> tuple[int, float]:
    rng = random.Random(seed)
    alice_bits = [rng.randint(0, 1) for _ in range(n)]
    alice_bases = [rng.choice("+x") for _ in range(n)]
    bob_bases = [rng.choice("+x") for _ in range(n)]
    bob_bits = []
    for bit, basis, bob_basis in zip(alice_bits, alice_bases, bob_bases):
        if eve:  # intercept-resend: Eve measures in a random basis and sends a new photon
            eve_basis = rng.choice("+x")
            bit, basis = measure(bit, basis, eve_basis, rng), eve_basis
        bob_bits.append(measure(bit, basis, bob_basis, rng))
    keep = [i for i in range(n) if alice_bases[i] == bob_bases[i]]
    errors = sum(alice_bits[i] != bob_bits[i] for i in keep)
    return len(keep), errors / len(keep)

for eve in (False, True):
    kept, error_rate = bb84(4000, eve)
    print(f"eavesdropper={eve!s:5}  sifted key {kept} bits, error rate {error_rate:.1%}")

Without Eve the sifted key is error-free. With Eve, about **25%** of the
sifted bits disagree: half the time she picks the wrong basis, and then Bob
gets a random bit. Alice and Bob compare a sample of their key; a high error
rate means "someone is listening — throw the key away".

In [ ]:
_, clean = bb84(4000, eve=False)
_, tapped = bb84(4000, eve=True)
assert clean == 0 and 0.2 < tapped < 0.3

## How this relates to pqc-scan

BB84 needs special hardware and dedicated links, so it does not replace the
algorithms on the internet. The practical path — and what pqc-scan helps
with — is replacing RSA/ECC in software with post-quantum algorithms such as
ML-KEM and ML-DSA.